# Cargue de Shapefiles de Inundaciones en Colombia\n\nShapefiles del fenómeno **La Niña** a escala 1:100.000 para los años 1988, 2000, 2011 y 2012.\n\n**Fuente:** IGAC / IDEAM  \n**CRS:** EPSG:4686 (MAGNA-SIRGAS)

In [1]:
import geopandas as gpd

base = r"C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output4\Indicadores"

inundacion_1988 = gpd.read_file(rf"{base}\shape 1988\Inundacion_Niña_100k_1988.shp")
inundacion_2000 = gpd.read_file(rf"{base}\shape 2000\Inundacion_Niña_100k_2000.shp")
inundacion_2011 = gpd.read_file(rf"{base}\shape 2011\Inundacion_Niña_100k_2011.shp")
inundacion_2012 = gpd.read_file(rf"{base}\shape 2012\Inundacion_Niña_100k_2012.shp")

for año, gdf in [(1988, inundacion_1988), (2000, inundacion_2000), (2011, inundacion_2011), (2012, inundacion_2012)]:
    print(f"--- {año} ---")
    print(f"  Filas: {len(gdf)} | CRS: {gdf.crs} | Geometría: {gdf.geom_type.unique().tolist()}")


--- 1988 ---
  Filas: 10988 | CRS: EPSG:4686 | Geometría: ['Polygon']
--- 2000 ---
  Filas: 11724 | CRS: EPSG:4686 | Geometría: ['Polygon']
--- 2011 ---
  Filas: 2517 | CRS: EPSG:4686 | Geometría: ['Polygon']
--- 2012 ---
  Filas: 2817 | CRS: EPSG:4686 | Geometría: ['Polygon']


In [2]:
# Cargar shapefile de municipios (polígonos)
# El .prj está como .prj.txt, por lo que geopandas no lo lee automáticamente → se asigna manualmente
base_mun = r"C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output4\shape_mun_col"
municipios = gpd.read_file(rf"{base_mun}\Muni.shp")
municipios = municipios.set_crs(epsg=4686)  # MAGNA-SIRGAS, mismo que los shapefiles de inundación

print(f"Municipios cargados: {len(municipios)} | CRS: {municipios.crs}")
municipios[['MunCodigo', 'MunNombre']].head()


Municipios cargados: 1122 | CRS: EPSG:4686


,MunCodigo,MunNombre
0,73504,ORTEGA
1,73675,SAN ANTONIO
2,25506,VENECIA
3,73226,CUNDAY
4,76834,TULUÁ


In [3]:
import pandas as pd

# CRS de referencia: EPSG:4686 (MAGNA-SIRGAS), ya asignado a municipios y a los floods
crs_ref = municipios.crs  # ahora es EPSG:4686 y no es None

floods = {
    1988: inundacion_1988.to_crs(crs_ref),
    2000: inundacion_2000.to_crs(crs_ref),
    2011: inundacion_2011.to_crs(crs_ref),
    2012: inundacion_2012.to_crs(crs_ref),
}

# Spatial join: qué municipios intersectan áreas inundadas en cada año
muns_afectados = {}
for año, flood in floods.items():
    joined = gpd.sjoin(
        municipios[['MunCodigo', 'MunNombre', 'geometry']],
        flood[['geometry']],
        how='inner',
        predicate='intersects'
    )
    muns_afectados[año] = set(joined['MunCodigo'].unique())
    print(f"{año}: {len(muns_afectados[año])} municipios afectados")

# Construir tabla binaria (todos los municipios)
tabla = municipios[['MunCodigo', 'MunNombre']].drop_duplicates().copy()
for año in [1988, 2000, 2011, 2012]:
    tabla[str(año)] = tabla['MunCodigo'].isin(muns_afectados[año]).astype(int)

# Frecuencia: proporción de años en que el municipio fue afectado
tabla['frecuencia'] = tabla[['1988', '2000', '2011', '2012']].mean(axis=1).round(2)

tabla = tabla.sort_values('frecuencia', ascending=False).reset_index(drop=True)
tabla = tabla.rename(columns={'MunCodigo': 'cod_divipola', 'MunNombre': 'municipio'})

print(f"\nMunicipios afectados al menos una vez: {(tabla['frecuencia'] > 0).sum()} / {len(tabla)}")
tabla[tabla['frecuencia'] > 0].head(20)


1988: 873 municipios afectados
2000: 883 municipios afectados
2011: 821 municipios afectados
2012: 775 municipios afectados

Municipios afectados al menos una vez: 922 / 1122


,cod_divipola,municipio,1988,2000,2011,2012,frecuencia
0,50370,URIBE,1,1,1,1,1.0
1,11001,BOGOTÁ D.C.,1,1,1,1,1.0
2,44847,URIBIA,1,1,1,1,1.0
3,44560,MANAURE,1,1,1,1,1.0
4,44001,RIOHACHA,1,1,1,1,1.0
5,47001,SANTA MARTA,1,1,1,1,1.0
6,44090,DIBULLA,1,1,1,1,1.0
7,08001,BARRANQUILLA,1,1,1,1,1.0
8,47745,SITIONUEVO,1,1,1,1,1.0
9,47189,CIÉNAGA,1,1,1,1,1.0


In [4]:
tabla["frecuencia"].value_counts()

frecuencia
1.00    754
0.00    200
0.50    106
0.75     31
0.25     31
Name: count, dtype: int64